# Revolut Fraud FC Case

## Data management

#### Imports

In [342]:
import pandas as pd
import numpy as np

df = pd.read_csv("fraud_prevention_data.csv")
df.head()

,USER_ID,TYPE,AMOUNT,CURRENCY,MERCHANT_COUNTRY,KYC,BIRTH_YEAR,COUNTRY,IS_FRAUD
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,3738,GBP,AUS,PASSED,1963,GB,False
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,588,GBP,CA,PASSED,1988,GB,False
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,1264,GBP,UKR,PASSED,1977,GB,False
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,66,GBP,CA,PASSED,1988,GB,False
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,968,GBP,NZL,FAILED,1992,GB,False


#### Treat Country Columns

##### All

In [343]:
# Import ISO country code table

country_codes = pd.read_excel("CountryCodesTable_ISO3166.xlsx")
country_codes["Numeric_3"] = (
    country_codes["Numeric"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

country_codes.head()

,Country,Alpha-2-code,Alpha-3-code,Numeric,Numeric_3
0,Afghanistan,AF,AFG,4,004
1,Albania,AL,ALB,8,008
2,Algeria,DZ,DZA,12,012
3,American Samoa,AS,ASM,16,016
4,Andorra,AD,AND,20,020


In [344]:
## Creating From/To Dictionary

iso2_to_country_name = dict(
    zip(country_codes["Alpha-2-code"], country_codes["Country"])
)

iso3_to_country_name = dict(
    zip(country_codes["Alpha-3-code"], country_codes["Country"])
)

numeric_to_country_name = dict(
    zip(country_codes["Numeric_3"], country_codes["Country"])
)

manual_country_mapping = {
    "ROM": "Romania",
    "BP0": "Hungary",
    "BP1": "Hungary",
    "BP2": "Hungary",
    "NSW": "New South Wales",
    "FL" : "Liechtenstein"
}

country_depara = {}
country_depara.update(iso2_to_country_name)
country_depara.update(iso3_to_country_name)
country_depara.update(numeric_to_country_name)
country_depara.update(manual_country_mapping)

country_depara

{'AF': 'Afghanistan',
 'AL': 'Albania',
 'DZ': 'Algeria',
 'AS': 'American Samoa',
 'AD': 'Andorra',
 'AO': 'Angola',
 'AI': 'Anguilla',
 'AQ': 'Antarctica',
 'AG': 'Antigua and Barbuda',
 'AR': 'Argentina',
 'AM': 'Armenia',
 'AW': 'Aruba',
 'AU': 'Australia',
 'AT': 'Austria',
 'AZ': 'Azerbaijan',
 'BS': 'Bahamas (the)',
 'BH': 'Bahrain',
 'BD': 'Bangladesh',
 'BB': 'Barbados',
 'BY': 'Belarus',
 'BE': 'Belgium',
 'BZ': 'Belize',
 'BJ': 'Benin',
 'BM': 'Bermuda',
 'BT': 'Bhutan',
 'BO': 'Bolivia (Plurinational State of)',
 'BQ': 'Bonaire, Sint Eustatius and Saba',
 'BA': 'Bosnia and Herzegovina',
 'BW': 'Botswana',
 'BV': 'Bouvet Island',
 'BR': 'Brazil',
 'IO': 'British Indian Ocean Territory (the)',
 'BN': 'Brunei Darussalam',
 'BG': 'Bulgaria',
 'BF': 'Burkina Faso',
 'BI': 'Burundi',
 'CV': 'Cabo Verde',
 'KH': 'Cambodia',
 'CM': 'Cameroon',
 'CA': 'Canada',
 'KY': 'Cayman Islands (the)',
 'CF': 'Central African Republic (the)',
 'TD': 'Chad',
 'CL': 'Chile',
 'CN': 'China',
 'CX

##### Country

In [345]:
# Clean COUNTRY
df["COUNTRY_CLEAN"] = (
    df["COUNTRY"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Apply De/Para to COUNTRY
df["COUNTRY_NAME"] = df["COUNTRY_CLEAN"].map(country_depara)

##### Merchant Country

In [346]:
# ------------------------------------------------------------
# Clean MERCHANT_COUNTRY with safer extraction logic
# ------------------------------------------------------------

# Identify original nulls first
original_null = df["MERCHANT_COUNTRY"].isna()

# Clean text
df["MERCHANT_COUNTRY_CLEAN"] = (
    df["MERCHANT_COUNTRY"]
    .astype("string")
    .str.replace("%20", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.upper()
)

# Start extracted column as real missing values
df["MERCHANT_COUNTRY_EXTRACTED"] = pd.NA

# 1. BP codes only when the full value is BP + digits
mask_bp = df["MERCHANT_COUNTRY_CLEAN"].str.fullmatch(r"BP\d+", na=False)

df.loc[mask_bp, "MERCHANT_COUNTRY_EXTRACTED"] = df.loc[
    mask_bp, "MERCHANT_COUNTRY_CLEAN"
]

# 2. Numeric ISO only when the full value is numeric
mask_numeric = df["MERCHANT_COUNTRY_CLEAN"].str.fullmatch(r"\d{1,3}", na=False)

df.loc[mask_numeric, "MERCHANT_COUNTRY_EXTRACTED"] = (
    df.loc[mask_numeric, "MERCHANT_COUNTRY_CLEAN"]
    .str.zfill(3)
)

# 3. Alpha country code only when it appears as a final separate token
# Example: "PARIS FRA" -> "FRA"
mask_alpha_final_token = df["MERCHANT_COUNTRY_CLEAN"].str.contains(
    r"(?:^| )[A-Z]{2,3}$",
    na=False,
    regex=True
)

df.loc[mask_alpha_final_token, "MERCHANT_COUNTRY_EXTRACTED"] = (
    df.loc[mask_alpha_final_token, "MERCHANT_COUNTRY_CLEAN"]
    .str.extract(r"(?:^| )([A-Z]{2,3})$")[0]
)

# 4. Apply De/Para mapping
df["MERCHANT_COUNTRY_NAME"] = df["MERCHANT_COUNTRY_EXTRACTED"].map(country_depara)

# 5. Original null merchant country = Online
df.loc[original_null, "MERCHANT_COUNTRY_NAME"] = "Online"

# 6. Not null but not mapped = Other
df.loc[
    (~original_null) & (df["MERCHANT_COUNTRY_NAME"].isna()),
    "MERCHANT_COUNTRY_NAME"
] = "Other"

In [347]:
merchant_country_validation = (
    df.groupby(
        [
            "MERCHANT_COUNTRY",
            "MERCHANT_COUNTRY_CLEAN",
            "MERCHANT_COUNTRY_EXTRACTED",
            "MERCHANT_COUNTRY_NAME"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

merchant_country_validation.tail(50)

,MERCHANT_COUNTRY,MERCHANT_COUNTRY_CLEAN,MERCHANT_COUNTRY_EXTRACTED,MERCHANT_COUNTRY_NAME,rows
92,482,482,482,Other,1
91,480,480,480,Mauritius,1
119,ARI,ARI,ARI,Other,1
76,350,350,350,Other,1
100,660,660,660,Anguilla,1
99,655,655,655,Other,1
98,643,643,643,Russian Federation (the),1
287,OWN,OWN,OWN,Other,1
286,ON,ON,ON,Other,1
284,OH,OH,OH,Other,1


In [348]:
columns_to_drop = [
    "COUNTRY",
    "COUNTRY_CLEAN",
    "MERCHANT_COUNTRY",
    "MERCHANT_COUNTRY_CLEAN",
    "MERCHANT_COUNTRY_EXTRACTED"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

df.head()

,USER_ID,TYPE,AMOUNT,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,3738,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,588,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,1264,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,66,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,968,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand


##### Currency Country

In [349]:
# ============================================================
# Currency-country consistency validation
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Currency to country / region mapping
# ------------------------------------------------------------

currency_country_map = {
    # Major single-country currencies
    "GBP": "United Kingdom of Great Britain and Northern Ireland (the)",
    "USD": "United States of America (the)",
    "JPY": "Japan",
    "CHF": "Switzerland",
    "NOK": "Norway",
    "SEK": "Sweden",
    "DKK": "Denmark",
    "PLN": "Poland",
    "CZK": "Czechia",
    "HUF": "Hungary",
    "RON": "Romania",
    "TRY": "Turkey",
    "AUD": "Australia",
    "CAD": "Canada",
    "HKD": "Hong Kong",
    "ILS": "Israel",
    "INR": "India",
    "NZD": "New Zealand",
    "SGD": "Singapore",
    "THB": "Thailand",
    "ZAR": "South Africa",
    "AED": "United Arab Emirates (the)",
    "QAR": "Qatar",
    "MAD": "Morocco",

    # Euro is multi-country
    "EUR": "Eurozone",

    # Crypto is not country-based
    "BTC": "Crypto",
    "ETH": "Crypto",
    "LTC": "Crypto",
    "XRP": "Crypto"
}

eurozone_countries = [
    "Austria",
    "Belgium",
    "Croatia",
    "Cyprus",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Ireland",
    "Italy",
    "Latvia",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Netherlands (the)",
    "Portugal",
    "Slovakia",
    "Slovenia",
    "Spain"
]

crypto_currencies = ["BTC", "ETH", "LTC", "XRP"]

# ------------------------------------------------------------
# 2. Create currency country / region column
# ------------------------------------------------------------

df["CURRENCY_COUNTRY_NAME"] = df["CURRENCY"].map(currency_country_map)

df.head()

,USER_ID,TYPE,AMOUNT,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,3738,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the)
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,588,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the)
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,1264,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the)
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,66,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the)
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,968,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the)


#### Cross-boarder flag

In [350]:
# ============================================================
# 1. Cross-border transaction
# Compare COUNTRY_NAME vs MERCHANT_COUNTRY_NAME
# ============================================================

df["MERCHANT_COUNTRY_COMPARISON"] = (
    df["MERCHANT_COUNTRY_NAME"]
    .fillna("Other")
    .astype(str)
    .str.strip()
)

df["MERCHANT_COUNTRY_COMPARISON"] = np.where(
    df["MERCHANT_COUNTRY_COMPARISON"].isin(["", "Other", "Online", "nan", "NaN", "None"]),
    "Other",
    df["MERCHANT_COUNTRY_COMPARISON"]
)

df["CROSS_BORDER_TRANSACTION"] = np.where(
    df["COUNTRY_NAME"] == df["MERCHANT_COUNTRY_COMPARISON"],
    False,
    True
)

In [351]:
columns_to_drop = [
    "MERCHANT_COUNTRY_COMPARISON"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

df.head()

,USER_ID,TYPE,AMOUNT,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,3738,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,588,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,1264,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,66,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,968,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True


#### Different current transaction

In [352]:
# ============================================================
# 2. Different currency transaction
# Compare COUNTRY_NAME vs CURRENCY_COUNTRY_NAME
# ============================================================

df["CURRENCY_COUNTRY_COMPARISON"] = (
    df["CURRENCY_COUNTRY_NAME"]
    .fillna("Other")
    .astype(str)
    .str.strip()
)

df["CURRENCY_COUNTRY_COMPARISON"] = np.where(
    df["CURRENCY_COUNTRY_COMPARISON"].isin(["", "Other", "Crypto", "nan", "NaN", "None"]),
    "Other",
    df["CURRENCY_COUNTRY_COMPARISON"]
)

df["DIFFERENT_CURRENCY_TRANSACTION"] = np.where(
    df["COUNTRY_NAME"] == df["CURRENCY_COUNTRY_COMPARISON"],
    False,
    True
)

In [353]:
columns_to_drop = [
    "CURRENCY_COUNTRY_COMPARISON"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

df.head()

,USER_ID,TYPE,AMOUNT,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,3738,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,588,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,1264,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,66,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,968,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False


#### Treating currency amounts

In [354]:
# ============================================================
# Fixed currency conversion to GBP
# Snapshot around 2026-06-02
# AMOUNT_GBP = AMOUNT * currency_to_gbp_rate
# ============================================================

import pandas as pd
import numpy as np

currency_to_gbp = {
    # Fiat currencies: direct value of 1 unit of currency in GBP
    "GBP": 1.000000,
    "EUR": 0.864793,
    "USD": 0.743269,
    "JPY": 0.004657,
    "CZK": 0.035614,
    "DKK": 0.115876,
    "HUF": 0.002434,
    "PLN": 0.204161,
    "RON": 0.164714,
    "SEK": 0.079890,
    "CHF": 0.945537,
    "NOK": 0.080147,
    "TRY": 0.016174,
    "AUD": 0.532365,
    "CAD": 0.537215,
    "HKD": 0.094807,
    "ILS": 0.263635,
    "INR": 0.007827,
    "NZD": 0.441316,
    "SGD": 0.581314,
    "THB": 0.022798,
    "ZAR": 0.045599,
    "AED": 0.202388,
    "QAR": 0.204195,
    "MAD": 0.080849,

    # Crypto currencies: direct value of 1 coin/token in GBP
    "BTC": 51529.69,
    "ETH": 1465.17,
    "LTC": 37.79,
    "XRP": 0.9393
}

currency_conversion_table_gbp = pd.DataFrame({
    "CURRENCY": list(currency_to_gbp.keys()),
    "currency_to_gbp_rate": list(currency_to_gbp.values())
}).sort_values("CURRENCY").reset_index(drop=True)

currency_conversion_table_gbp

,CURRENCY,currency_to_gbp_rate
0,AED,0.20
1,AUD,0.53
2,BTC,"51,529.69"
3,CAD,0.54
4,CHF,0.95
5,CZK,0.04
6,DKK,0.12
7,ETH,"1,465.17"
8,EUR,0.86
9,GBP,1.00


In [355]:
## Mapping and converting the currency

df["currency_to_gbp_rate"] = df["CURRENCY"].map(currency_to_gbp)

df["AMOUNT_GBP"] = df["AMOUNT"] * df["currency_to_gbp_rate"]

## Creating Fiat or Crypto column
crypto_currencies = ["BTC", "ETH", "LTC", "XRP"]

df["CURRENCY_TYPE"] = np.where(
    df["CURRENCY"].isin(crypto_currencies),
    "CRYPTO",
    "FIAT"
)

df[["CURRENCY", "CURRENCY_TYPE"]].drop_duplicates().sort_values("CURRENCY")

df.head()

,USER_ID,TYPE,AMOUNT,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,currency_to_gbp_rate,AMOUNT_GBP,CURRENCY_TYPE
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,3738,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False,1.00,"3,738.00",FIAT
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,588,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,1.00,588.00,FIAT
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,1264,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False,1.00,"1,264.00",FIAT
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,66,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,1.00,66.00,FIAT
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,968,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False,1.00,968.00,FIAT


In [356]:
currency_validation = df.groupby("CURRENCY").agg(
    transactions=("USER_ID", "count"),
    original_amount=("AMOUNT", "sum"),
    amount_gbp=("AMOUNT_GBP", "sum"),
    avg_fx_rate=("currency_to_gbp_rate", "mean"),
    min_fx_rate=("currency_to_gbp_rate", "min"),
    max_fx_rate=("currency_to_gbp_rate", "max")
).reset_index()

currency_validation["implied_fx_rate"] = (
    currency_validation["amount_gbp"] /
    currency_validation["original_amount"]
).replace([np.inf, -np.inf], 0).fillna(0)

currency_validation = currency_validation.sort_values(
    "amount_gbp",
    ascending=False
)

currency_validation

,CURRENCY,transactions,original_amount,amount_gbp,avg_fx_rate,min_fx_rate,max_fx_rate,implied_fx_rate
2,BTC,283,468004199,"24,116,111,293,168.31","51,529.69","51,529.69","51,529.69","51,529.69"
7,ETH,197,1638854181,"2,401,199,980,375.77","1,465.17","1,465.17","1,465.17","1,465.17"
15,LTC,137,5620383823,"212,394,304,671.17",37.79,37.79,37.79,37.79
9,GBP,339091,2874187870,"2,874,187,870.00",1.00,1.00,1.00,1.00
27,XRP,38,2525590470,"2,372,287,128.47",0.94,0.94,0.94,0.94
8,EUR,264695,2164849186,"1,872,146,422.11",0.86,0.86,0.86,0.86
26,USD,31542,365847921,"271,923,418.39",0.74,0.74,0.74,0.74
19,PLN,22362,536371138,"109,506,067.91",0.20,0.20,0.20,0.20
4,CHF,5761,112040047,"105,938,009.92",0.95,0.95,0.95,0.95
21,RON,5837,142754858,"23,513,723.68",0.16,0.16,0.16,0.16


In [357]:
unmapped_currencies = (
    df.loc[df["currency_to_gbp_rate"].isna(), "CURRENCY"]
    .drop_duplicates()
    .sort_values()
)

print("Unmapped currencies:")
print(unmapped_currencies)

Unmapped currencies:
Series([], Name: CURRENCY, dtype: object)


In [358]:
columns_to_drop = [
    "AMOUNT",
    "currency_to_gbp_rate"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

df.head()

,USER_ID,TYPE,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,AMOUNT_GBP,CURRENCY_TYPE
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False,"3,738.00",FIAT
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,588.00,FIAT
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False,"1,264.00",FIAT
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,66.00,FIAT
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False,968.00,FIAT


#### Creating outlier amount GBP transaction

In [359]:
# Calculate IQR limits for AMOUNT_GBP
q1 = df["AMOUNT_GBP"].quantile(0.20)
q3 = df["AMOUNT_GBP"].quantile(0.80)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Create outlier flag
df["FLAG_OUTLIER"] = np.where(
    (df["AMOUNT_GBP"] < lower_bound) | 
    (df["AMOUNT_GBP"] > upper_bound),
    True,
    False
)

print(f"Q1: {q1:,.2f}")
print(f"Q3: {q3:,.2f}")
print(f"IQR: {iqr:,.2f}")
print(f"Lower bound: {lower_bound:,.2f}")
print(f"Upper bound: {upper_bound:,.2f}")

df["FLAG_OUTLIER"].value_counts(dropna=False)

Q1: 290.00
Q3: 5,392.63
IQR: 5,102.63
Lower bound: -7,363.95
Upper bound: 13,046.58


FLAG_OUTLIER
False    620444
True      68207
Name: count, dtype: int64

In [360]:
outlier_summary = (
    df.groupby(["CURRENCY_TYPE", "IS_FRAUD"], dropna=False)
    .agg(
        transactions=("USER_ID", "count")
    )
    .reset_index()
)

outlier_summary["pct_total"] = (
    outlier_summary["transactions"] / len(df) * 100
).round(2)

outlier_summary = outlier_summary.sort_values(
    ["CURRENCY_TYPE"]
)

outlier_summary

,CURRENCY_TYPE,IS_FRAUD,transactions,pct_total
0,CRYPTO,False,652,0.09
1,CRYPTO,True,3,0.00
2,FIAT,False,673456,97.79
3,FIAT,True,14540,2.11


In [361]:
df[df["FLAG_OUTLIER"] == True].sort_values(
    "AMOUNT_GBP",
    ascending=False
).head(50)

,USER_ID,TYPE,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,AMOUNT_GBP,CURRENCY_TYPE,FLAG_OUTLIER
359878,e908e675-7c4c-41a0-9c2d-b903a1a74c8a,P2P,BTC,PASSED,1986,False,Cyprus,Online,Crypto,True,True,"10,305,938,000,000.00",CRYPTO,True
496823,d2492abc-1b91-4fc4-aed2-2bcbd06319aa,P2P,BTC,PASSED,1982,False,Latvia,Online,Crypto,True,True,"5,152,969,000,000.00",CRYPTO,True
576573,3266d82b-2f97-474a-a581-c9425a6383d7,CARD_PAYMENT,BTC,PASSED,1983,False,France,United Kingdom of Great Britain and Northern Ireland (the),Crypto,True,True,"673,093,590,143.12",CRYPTO,True
493796,3fc813b5-39dc-4780-bd4f-bd7f9c6fdb5b,CARD_PAYMENT,BTC,PASSED,1993,False,United Kingdom of Great Britain and Northern Ireland (the),United Kingdom of Great Britain and Northern Ireland (the),Crypto,False,True,"277,804,236,713.81",CRYPTO,True
360850,605452f6-e9ce-42ef-bbf4-71e47958dcee,CARD_PAYMENT,BTC,PASSED,1990,False,United Kingdom of Great Britain and Northern Ireland (the),United Kingdom of Great Britain and Northern Ireland (the),Crypto,False,True,"254,693,737,575.40",CRYPTO,True
623159,189b675b-b6cf-4375-b906-058d8dd0eb68,CARD_PAYMENT,BTC,PASSED,1983,False,Switzerland,United Kingdom of Great Britain and Northern Ireland (the),Crypto,True,True,"239,439,558,033.77",CRYPTO,True
623197,189b675b-b6cf-4375-b906-058d8dd0eb68,CARD_PAYMENT,BTC,PASSED,1983,False,Switzerland,United Kingdom of Great Britain and Northern Ireland (the),Crypto,True,True,"239,425,438,898.71",CRYPTO,True
368549,605452f6-e9ce-42ef-bbf4-71e47958dcee,CARD_PAYMENT,BTC,PASSED,1990,False,United Kingdom of Great Britain and Northern Ireland (the),Poland,Crypto,True,True,"203,087,835,163.89",CRYPTO,True
493795,3fc813b5-39dc-4780-bd4f-bd7f9c6fdb5b,CARD_PAYMENT,BTC,PASSED,1993,False,United Kingdom of Great Britain and Northern Ireland (the),United Kingdom of Great Britain and Northern Ireland (the),Crypto,False,True,"185,217,235,612.51",CRYPTO,True
368789,605452f6-e9ce-42ef-bbf4-71e47958dcee,CARD_PAYMENT,BTC,PASSED,1990,False,United Kingdom of Great Britain and Northern Ireland (the),United Kingdom of Great Britain and Northern Ireland (the),Crypto,False,True,"175,681,202,710.80",CRYPTO,True


In [362]:
df.head()

,USER_ID,TYPE,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,AMOUNT_GBP,CURRENCY_TYPE,FLAG_OUTLIER
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False,"3,738.00",FIAT,False
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,588.00,FIAT,False
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False,"1,264.00",FIAT,False
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,66.00,FIAT,False
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False,968.00,FIAT,False


#### Creating count similar values column

In [363]:
# Count transactions with the same AMOUNT_GBP within each USER_ID
df["SAME_AMOUNT_GBP_TX_COUNT_BY_USER"] = (
    df.groupby(["USER_ID", "AMOUNT_GBP"])["USER_ID"]
    .transform("count")
)

# Count transactions with the same AMOUNT_GBP within each USER_ID and TYPE
df["SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE"] = (
    df.groupby(["USER_ID", "TYPE", "AMOUNT_GBP"])["USER_ID"]
    .transform("count")
)

df.head()

,USER_ID,TYPE,CURRENCY,KYC,BIRTH_YEAR,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,AMOUNT_GBP,CURRENCY_TYPE,FLAG_OUTLIER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,GBP,PASSED,1963,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False,"3,738.00",FIAT,False,1,1
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,588.00,FIAT,False,1,1
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,GBP,PASSED,1977,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False,"1,264.00",FIAT,False,1,1
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,1988,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,66.00,FIAT,False,1,1
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,GBP,FAILED,1992,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False,968.00,FIAT,False,1,1


In [364]:
same_amount_validation = (
    df.groupby("SAME_AMOUNT_GBP_TX_COUNT_BY_USER")
    .agg(
        transactions=("USER_ID", "count"),
        unique_users=("USER_ID", "nunique"),
        fraud_transactions=("IS_FRAUD", "sum")
    )
    .reset_index()
    .sort_values("SAME_AMOUNT_GBP_TX_COUNT_BY_USER", ascending=False)
)

same_amount_validation["fraud_rate_pct"] = (
    same_amount_validation["fraud_transactions"] /
    same_amount_validation["transactions"] * 100
).round(2)

same_amount_validation.head()

,SAME_AMOUNT_GBP_TX_COUNT_BY_USER,transactions,unique_users,fraud_transactions,fraud_rate_pct
128,298,298,1,0,0.00
127,279,279,1,0,0.00
126,241,241,1,0,0.00
125,198,396,2,0,0.00
124,187,187,1,0,0.00


In [365]:
same_amount_type_count_validation = (
    df.groupby("SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE")
    .agg(
        transactions=("USER_ID", "count"),
        unique_users=("USER_ID", "nunique"),
        fraud_transactions=("IS_FRAUD", "sum")
    )
    .reset_index()
    .sort_values("SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE", ascending=False)
)

same_amount_type_count_validation["fraud_rate_pct"] = (
    same_amount_type_count_validation["fraud_transactions"] /
    same_amount_type_count_validation["transactions"] * 100
).round(2)

same_amount_type_count_validation.head()

,SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE,transactions,unique_users,fraud_transactions,fraud_rate_pct
112,269,269,1,0,0.00
111,198,198,1,0,0.00
110,193,193,1,0,0.00
109,187,187,1,0,0.00
108,165,165,1,0,0.00


#### Create Age column

In [366]:
df["AGE"] = 2026 - df["BIRTH_YEAR"]

In [367]:
columns_to_drop = [
    "BIRTH_YEAR"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

df.head()

,USER_ID,TYPE,CURRENCY,KYC,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,AMOUNT_GBP,CURRENCY_TYPE,FLAG_OUTLIER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE,AGE
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False,"3,738.00",FIAT,False,1,1,63
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,588.00,FIAT,False,1,1,38
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False,"1,264.00",FIAT,False,1,1,49
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,66.00,FIAT,False,1,1,38
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,GBP,FAILED,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False,968.00,FIAT,False,1,1,34


#### Final dataset print

In [368]:
df.head(10)

,USER_ID,TYPE,CURRENCY,KYC,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,AMOUNT_GBP,CURRENCY_TYPE,FLAG_OUTLIER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE,AGE
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False,"3,738.00",FIAT,False,1,1,63
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,588.00,FIAT,False,1,1,38
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False,"1,264.00",FIAT,False,1,1,49
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,66.00,FIAT,False,1,1,38
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,GBP,FAILED,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False,968.00,FIAT,False,1,1,34
5,fbe6dfd9-96de-4fde-af16-32e8a0bb7a25,CARD_PAYMENT,GBP,NONE,False,United Kingdom of Great Britain and Northern Ireland (the),Online,United Kingdom of Great Britain and Northern Ireland (the),True,False,"1,641.00",FIAT,False,1,1,46
6,dd1f6199-127f-49ff-ba25-81393b2e66f2,CARD_PAYMENT,USD,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Liechtenstein,United States of America (the),True,True,"4,835.71",FIAT,False,1,1,50
7,dd1f6199-127f-49ff-ba25-81393b2e66f2,CARD_PAYMENT,USD,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Liechtenstein,United States of America (the),True,True,"7,204.51",FIAT,False,1,1,50
8,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),United Kingdom of Great Britain and Northern Ireland (the),United Kingdom of Great Britain and Northern Ireland (the),False,False,295.00,FIAT,False,2,2,38
9,1a9e22bd-4cec-47c4-8196-92038ba2e603,CARD_PAYMENT,EUR,PASSED,False,Greece,Luxembourg,Eurozone,True,True,76.97,FIAT,False,1,1,51


## Missions

#### Brief 1

In [397]:
# Step 1: make a TOPUP
step_1_users = set(
    df.loc[
        df["TYPE"] == "TOPUP",
        "USER_ID"
    ].unique()
)

# Step 2: make a TOPUP + KYC PASSED
step_2_users = set(
    df.loc[
        (df["TYPE"] == "TOPUP") &
        (df["KYC"] == "PASSED"),
        "USER_ID"
    ].unique()
)

# Step 3: make a TOPUP + fraud false
step_3_users = set(
    df.loc[
        (df["TYPE"] == "TOPUP") &
        (df["IS_FRAUD"] == False),
        "USER_ID"
    ].unique()
)

# Step 4: make a TOPUP + KYC PASSED + fraud false
step_4_users = set(
    df.loc[
        (df["TYPE"] == "TOPUP") &
        (df["KYC"] == "PASSED") &
        (df["IS_FRAUD"] == False),
        "USER_ID"
    ].unique()
)

# Users with other transaction excluding TOPUP
other_excl_topup_users = set(
    df.loc[
        df["TYPE"] != "TOPUP",
        "USER_ID"
    ].unique()
)

# Users with other transaction excluding TOPUP and ATM
other_excl_topup_atm_users = set(
    df.loc[
        ~df["TYPE"].isin(["TOPUP", "ATM"]),
        "USER_ID"
    ].unique()
)

# Step 5: TOPUP + other transaction excluding TOPUP
step_5_users = step_1_users.intersection(other_excl_topup_users)

# Step 6: TOPUP KYC PASSED + other transaction excluding TOPUP
step_6_users = step_2_users.intersection(other_excl_topup_users)

# Step 7: TOPUP fraud false + other transaction excluding TOPUP
step_7_users = step_3_users.intersection(other_excl_topup_users)

# Step 8: TOPUP KYC PASSED fraud false + other transaction excluding TOPUP
step_8_users = step_4_users.intersection(other_excl_topup_users)

# Step 9: TOPUP + other transaction excluding TOPUP and ATM
step_9_users = step_1_users.intersection(other_excl_topup_atm_users)

# Step 10: TOPUP KYC PASSED + other transaction excluding TOPUP and ATM
step_10_users = step_2_users.intersection(other_excl_topup_atm_users)

# Step 11: TOPUP fraud false + other transaction excluding TOPUP and ATM
step_11_users = step_3_users.intersection(other_excl_topup_atm_users)

# Step 12: TOPUP KYC PASSED fraud false + other transaction excluding TOPUP and ATM
step_12_users = step_4_users.intersection(other_excl_topup_atm_users)

# Summary table
app_conversion_funnel = pd.DataFrame({
    "step": [
        "1. TOPUP",
        "2. TOPUP + KYC PASSED",
        "3. TOPUP + fraud false",
        "4. TOPUP + KYC PASSED + fraud false",
        "5. TOPUP + other transaction excluding TOPUP",
        "6. TOPUP KYC PASSED + other transaction excluding TOPUP",
        "7. TOPUP fraud false + other transaction excluding TOPUP",
        "8. TOPUP KYC PASSED fraud false + other transaction excluding TOPUP",
        "9. TOPUP + other transaction excluding TOPUP and ATM",
        "10. TOPUP KYC PASSED + other transaction excluding TOPUP and ATM",
        "11. TOPUP fraud false + other transaction excluding TOPUP and ATM",
        "12. TOPUP KYC PASSED fraud false + other transaction excluding TOPUP and ATM"
    ],
    "unique_users": [
        len(step_1_users),
        len(step_2_users),
        len(step_3_users),
        len(step_4_users),
        len(step_5_users),
        len(step_6_users),
        len(step_7_users),
        len(step_8_users),
        len(step_9_users),
        len(step_10_users),
        len(step_11_users),
        len(step_12_users)
    ]
})

app_conversion_funnel["conversion_vs_step_1_pct"] = (
    app_conversion_funnel["unique_users"] / len(step_1_users) * 100
).round(2)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

app_conversion_funnel

,step,unique_users,conversion_vs_step_1_pct
0,1. TOPUP,7764,100.00
1,2. TOPUP + KYC PASSED,6878,88.59
2,3. TOPUP + fraud false,7465,96.15
3,4. TOPUP + KYC PASSED + fraud false,6618,85.24
4,5. TOPUP + other transaction excluding TOPUP,6394,82.35
5,6. TOPUP KYC PASSED + other transaction excluding TOPUP,6139,79.07
6,7. TOPUP fraud false + other transaction excluding TOPUP,6102,78.59
7,8. TOPUP KYC PASSED fraud false + other transaction excluding TOPUP,5885,75.80
8,9. TOPUP + other transaction excluding TOPUP and ATM,6311,81.29
9,10. TOPUP KYC PASSED + other transaction excluding TOPUP and ATM,6068,78.16


#### Brief 2

##### Part A

In [247]:
country_rates = df.groupby("COUNTRY_NAME", dropna=False).agg(
    transactions=("USER_ID", "count"),
    users=("USER_ID", "nunique"),
    total_amount=("AMOUNT_GBP", "sum"),
    fraud_transactions=("IS_FRAUD", "sum"),
    fraudster_users=("USER_ID", lambda x: df.loc[x.index][df.loc[x.index, "IS_FRAUD"]]["USER_ID"].nunique()),
    fraud_amount=("AMOUNT_GBP", lambda x: x[df.loc[x.index, "IS_FRAUD"]].sum()),
    kyc_passed_transactions=("KYC", lambda x: (x == "PASSED").sum())
).reset_index()

country_rates["fraud_rate_pct"] = (
    country_rates["fraud_transactions"] / country_rates["transactions"] * 100
).round(2)

country_rates["fraud_user_rate_pct"] = (
    country_rates["fraudster_users"] / country_rates["users"] * 100
).round(2)

country_rates["fraud_amount_rate_pct"] = (
    country_rates["fraud_amount"] / country_rates["total_amount"] * 100
).round(2)

country_rates["kyc_passed_rate_pct"] = (
    country_rates["kyc_passed_transactions"] / country_rates["transactions"] * 100
).round(2)

country_rates = country_rates.sort_values("fraud_amount", ascending=False)

pd.set_option("display.float_format", "{:,.2f}".format)

country_rates[["COUNTRY_NAME", "transactions", "users", "fraudster_users" , "fraud_amount", "fraud_rate_pct", "fraud_user_rate_pct", "fraud_amount_rate_pct"]]

,COUNTRY_NAME,transactions,users,fraudster_users,fraud_amount,fraud_rate_pct,fraud_user_rate_pct,fraud_amount_rate_pct
54,United Kingdom of Great Britain and Northern Ireland (the),385343,3898,269,"2,621,539,437.58",3.40,6.90,0.03
49,Spain,19510,298,3,"9,471,511.11",0.80,1.01,0.01
38,Poland,25725,553,2,"8,787,649.22",1.35,0.36,0.05
12,France,60184,951,3,"3,802,602.81",0.35,0.32,0.00
15,Germany,9012,148,1,"2,814,923.47",5.64,0.68,0.00
42,Romania,6903,155,4,"2,257,551.71",0.84,2.58,0.00
30,Lithuania,50725,363,14,"859,036.29",0.31,3.86,0.00
8,Czechia,2265,79,1,"138,081.00",0.49,1.27,0.00
3,Belgium,2919,57,1,"114,060.33",0.14,1.75,0.00
35,Netherlands (the),6102,84,1,"81,947.59",0.08,1.19,0.10


In [95]:
uk_name = "United Kingdom of Great Britain and Northern Ireland (the)"

uk_share_total = pd.DataFrame({
    "metric": [
        "fraud_transactions",
        "fraud_amount",
        "fraudster_users"
    ],
    "uk_value": [
        country_rates.loc[country_rates["COUNTRY_NAME"] == uk_name, "fraud_transactions"].sum(),
        country_rates.loc[country_rates["COUNTRY_NAME"] == uk_name, "fraud_amount"].sum(),
        country_rates.loc[country_rates["COUNTRY_NAME"] == uk_name, "fraudster_users"].sum()
    ],
    "total_value": [
        country_rates["fraud_transactions"].sum(),
        country_rates["fraud_amount"].sum(),
        country_rates["fraudster_users"].sum()
    ]
})

uk_share_total["uk_share_of_total_pct"] = (
    uk_share_total["uk_value"] / uk_share_total["total_value"] * 100
).round(2)

uk_share_total

,metric,uk_value,total_value,uk_share_of_total_pct
0,fraud_transactions,"13,088.00","14,543.00",90.00
1,fraud_amount,"2,621,539,437.58","2,649,866,801.11",98.93
2,fraudster_users,269.00,299.00,89.97


##### Part B

In [379]:
# ============================================================
# User-level fraud behaviour model
# Target: IS_FRAUD_USER
# Population: KYC PASSED users
# Includes repeated amount features
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# ============================================================
# 1. Keep only KYC PASSED population
# Business question: fraudsters who passed KYC
# ============================================================

df_kyc_passed = df[df["KYC"] == "PASSED"].copy()

# ============================================================
# 2. Build user-level dataset
# ============================================================

user_model_df = df_kyc_passed.groupby("USER_ID").agg(
    IS_FRAUD_USER=("IS_FRAUD", "max"),

    AGE=("AGE", "first"),
    COUNTRY_NAME=("COUNTRY_NAME", "first"),

    total_transactions=("USER_ID", "count"),

    avg_amount_gbp=("AMOUNT_GBP", "mean"),
    #median_amount_gbp=("AMOUNT_GBP", "median"),
    #max_amount_gbp=("AMOUNT_GBP", "max"),

    unique_types=("TYPE", "nunique"),
    unique_currencies=("CURRENCY", "nunique"),
    unique_merchant_countries=("MERCHANT_COUNTRY_NAME", "nunique"),

    cross_border_transactions=("CROSS_BORDER_TRANSACTION", "sum"),
    different_currency_transactions=("DIFFERENT_CURRENCY_TRANSACTION", "sum"),
    outlier_transactions=("FLAG_OUTLIER", "sum"),

    #avg_same_amount_count_by_user=("SAME_AMOUNT_GBP_TX_COUNT_BY_USER", "mean"),
    max_same_amount_count_by_user=("SAME_AMOUNT_GBP_TX_COUNT_BY_USER", "max"),
    #median_same_amount_count_by_user=("SAME_AMOUNT_GBP_TX_COUNT_BY_USER", "median"),

    #avg_same_amount_count_by_user_type=("SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE", "mean"),
    max_same_amount_count_by_user_type=("SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE", "max"),
    #median_same_amount_count_by_user_type=("SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE", "median")
).reset_index()

# ============================================================
# 3. Behavioural shares
# ============================================================

user_model_df["cross_border_share"] = (
    user_model_df["cross_border_transactions"] /
    user_model_df["total_transactions"]
)

user_model_df["different_currency_share"] = (
    user_model_df["different_currency_transactions"] /
    user_model_df["total_transactions"]
)

user_model_df["outlier_share"] = (
    user_model_df["outlier_transactions"] /
    user_model_df["total_transactions"]
)

# ============================================================
# 4. Transaction type share per user
# ============================================================

type_share = (
    pd.crosstab(
        df_kyc_passed["USER_ID"],
        df_kyc_passed["TYPE"],
        normalize="index"
    )
    .add_prefix("share_type_")
    .reset_index()
)

user_model_df = user_model_df.merge(
    type_share,
    on="USER_ID",
    how="left"
)

# ============================================================
# 5. Currency share per user
# ============================================================

currency_share = (
    pd.crosstab(
        df_kyc_passed["USER_ID"],
        df_kyc_passed["CURRENCY"],
        normalize="index"
    )
    .add_prefix("share_currency_")
    .reset_index()
)

user_model_df = user_model_df.merge(
    currency_share,
    on="USER_ID",
    how="left"
)

# ============================================================
# 6. Currency type share per user: FIAT vs CRYPTO
# ============================================================

currency_type_share = (
    pd.crosstab(
        df_kyc_passed["USER_ID"],
        df_kyc_passed["CURRENCY_TYPE"],
        normalize="index"
    )
    .add_prefix("share_currency_type_")
    .reset_index()
)

user_model_df = user_model_df.merge(
    currency_type_share,
    on="USER_ID",
    how="left"
)

# Fill missing share columns with zero
user_model_df = user_model_df.fillna(0)

# ============================================================
# 7. Create COUNTRY_SEGMENT: UK vs NON_UK
# ============================================================

uk_country_name = "United Kingdom of Great Britain and Northern Ireland (the)"

user_model_df["COUNTRY_SEGMENT"] = np.where(
    user_model_df["COUNTRY_NAME"] == uk_country_name,
    "UK",
    "NON_UK"
)

# ============================================================
# 8. Define model features
# ============================================================

features_user = [
    "AGE",
    "COUNTRY_NAME",

    # Keep commented to avoid pure volume bias
    # "total_transactions",

    "avg_amount_gbp",
    #"median_amount_gbp",
    #"max_amount_gbp",

    "unique_types",
    "unique_currencies",
    "unique_merchant_countries",

    "cross_border_share",
    "different_currency_share",
    "outlier_share",

    #"avg_same_amount_count_by_user",
    "max_same_amount_count_by_user",
    #"median_same_amount_count_by_user",

    #"avg_same_amount_count_by_user_type",
    "max_same_amount_count_by_user_type",
    #"median_same_amount_count_by_user_type"
] + [
    col for col in user_model_df.columns
    if (
        col.startswith("share_type_") or
        col.startswith("share_currency_") or
        col.startswith("share_currency_type_")
    )
]

# ============================================================
# 9. Function to train one model per country segment
# ============================================================

def train_user_level_model_by_country_segment(segment_name):
    
    segment_df = user_model_df[
        user_model_df["COUNTRY_SEGMENT"] == segment_name
    ].copy()
    
    X_segment = segment_df[features_user]
    y_segment = segment_df["IS_FRAUD_USER"]
    
    print("\n================================================")
    print(f"Country segment: {segment_name}")
    print(f"Users: {segment_df['USER_ID'].nunique()}")
    print(f"Fraudster users: {int(y_segment.sum())}")
    print(f"Fraudster rate: {y_segment.mean() * 100:.2f}%")
    print("================================================")
    
    print("\nTarget distribution:")
    print(y_segment.value_counts(dropna=False))
    
    if y_segment.nunique() < 2:
        print(f"\nSkipping {segment_name}: only one target class available.")
        return None, None, segment_df
    
    min_class_count = y_segment.value_counts().min()
    stratify_target = y_segment if min_class_count >= 2 else None
    
    categorical_features_segment = ["COUNTRY_NAME"]
    
    numeric_features_segment = [
        col for col in features_user
        if col not in categorical_features_segment
    ]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_segment,
        y_segment,
        test_size=0.30,
        random_state=42,
        stratify=stratify_target
    )
    
    preprocessor_segment = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_segment),
            ("num", "passthrough", numeric_features_segment)
        ]
    )
    
    model_segment = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42
    )
    
    pipeline_segment = Pipeline(
        steps=[
            ("preprocessor", preprocessor_segment),
            ("model", model_segment)
        ]
    )
    
    pipeline_segment.fit(X_train, y_train)
    
    y_pred = pipeline_segment.predict(X_test)
    y_proba = pipeline_segment.predict_proba(X_test)[:, 1]
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nROC AUC:")
    print(roc_auc_score(y_test, y_proba))
    
    encoded_feature_names = pipeline_segment.named_steps["preprocessor"].get_feature_names_out()
    
    feature_importance_segment = pd.DataFrame({
        "country_segment": segment_name,
        "feature": encoded_feature_names,
        "importance": pipeline_segment.named_steps["model"].feature_importances_
    })
    
    feature_importance_segment["importance_pct"] = (
        feature_importance_segment["importance"] * 100
    ).round(2)
    
    feature_importance_segment = feature_importance_segment.sort_values(
        "importance_pct",
        ascending=False
    ).reset_index(drop=True)
    
    return pipeline_segment, feature_importance_segment, segment_df

# ============================================================
# 10. Train UK and NON_UK models
# ============================================================

uk_pipeline, uk_feature_importance, uk_user_model_df = train_user_level_model_by_country_segment("UK")

non_uk_pipeline, non_uk_feature_importance, non_uk_user_model_df = train_user_level_model_by_country_segment("NON_UK")

# ============================================================
# 11. Display feature importance
# ============================================================

print("\nUK feature importance:")
display(uk_feature_importance.head(30))

print("\nNON-UK feature importance:")
display(non_uk_feature_importance.head(30))


Country segment: UK
Users: 3438
Fraudster users: 231
Fraudster rate: 6.72%

Target distribution:
IS_FRAUD_USER
False    3207
True      231
Name: count, dtype: int64

Confusion Matrix:
[[831 132]
 [ 27  42]]

Classification Report:
              precision    recall  f1-score   support

       False       0.97      0.86      0.91       963
        True       0.24      0.61      0.35        69

    accuracy                           0.85      1032
   macro avg       0.60      0.74      0.63      1032
weighted avg       0.92      0.85      0.87      1032


ROC AUC:
0.8387286107724954

Country segment: NON_UK
Users: 3551
Fraudster users: 29
Fraudster rate: 0.82%

Target distribution:
IS_FRAUD_USER
False    3522
True       29
Name: count, dtype: int64

Confusion Matrix:
[[999  58]
 [  2   7]]

Classification Report:
              precision    recall  f1-score   support

       False       1.00      0.95      0.97      1057
        True       0.11      0.78      0.19         9

    accuracy 

,country_segment,feature,importance,importance_pct
0,UK,num__avg_amount_gbp,0.46,45.74
1,UK,num__share_currency_GBP,0.28,27.68
2,UK,num__share_type_BANK_TRANSFER,0.10,9.81
3,UK,num__outlier_share,0.05,4.87
4,UK,num__share_type_CARD_PAYMENT,0.04,3.64
5,UK,num__cross_border_share,0.03,3.09
6,UK,num__unique_merchant_countries,0.03,2.64
7,UK,num__AGE,0.02,1.71
8,UK,num__share_currency_CHF,0.01,0.82
9,UK,num__unique_types,0.00,0.00



NON-UK feature importance:


,country_segment,feature,importance,importance_pct
0,NON_UK,num__share_currency_GBP,0.73,72.77
1,NON_UK,num__avg_amount_gbp,0.14,13.98
2,NON_UK,num__max_same_amount_count_by_user,0.10,10.01
3,NON_UK,num__AGE,0.03,3.23
4,NON_UK,cat__COUNTRY_NAME_Austria,0.00,0.00
5,NON_UK,cat__COUNTRY_NAME_Belgium,0.00,0.00
6,NON_UK,cat__COUNTRY_NAME_Denmark,0.00,0.00
7,NON_UK,cat__COUNTRY_NAME_Estonia,0.00,0.00
8,NON_UK,cat__COUNTRY_NAME_Finland,0.00,0.00
9,NON_UK,cat__COUNTRY_NAME_France,0.00,0.00


In [380]:
# ============================================================
# Business descriptive view by UK / NON_UK segment
# Features selected when importance_pct > 0.01%
# ============================================================

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

importance_pct_threshold = 0.01

# ============================================================
# 1. Combine feature importance from UK and NON_UK models
# ============================================================

combined_feature_importance = pd.concat(
    [
        uk_feature_importance.copy(),
        non_uk_feature_importance.copy()
    ],
    ignore_index=True
)

# Clean feature names from pipeline prefixes
combined_feature_importance["clean_feature"] = (
    combined_feature_importance["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

# Keep only features with importance_pct > 0.01%
relevant_features_by_segment = combined_feature_importance[
    combined_feature_importance["importance_pct"] > importance_pct_threshold
].copy()

# ============================================================
# 2. Dictionary with the correct user-level dataframe by segment
# ============================================================

segment_user_model_dfs = {
    "UK": uk_user_model_df.copy(),
    "NON_UK": non_uk_user_model_df.copy()
}

# ============================================================
# 3. Keep only features that exist as direct columns
# This avoids one-hot categorical features like COUNTRY_NAME_Germany
# ============================================================

valid_rows = []

for _, row in relevant_features_by_segment.iterrows():
    segment = row["country_segment"]
    feature = row["clean_feature"]
    
    if feature in segment_user_model_dfs[segment].columns:
        valid_rows.append(row)

relevant_features_by_segment = pd.DataFrame(valid_rows)

# ============================================================
# 4. Build descriptive table
# ============================================================

feature_descriptions = []

for _, row in relevant_features_by_segment.iterrows():
    segment = row["country_segment"]
    feature = row["clean_feature"]
    importance = row["importance"]
    importance_pct = row["importance_pct"]
    
    segment_df = segment_user_model_dfs[segment]
    
    desc = (
        segment_df
        .groupby("IS_FRAUD_USER")[feature]
        .describe()
        .reset_index()
    )
    
    desc.insert(0, "country_segment", segment)
    desc.insert(1, "feature", feature)
    desc.insert(2, "importance", importance)
    desc.insert(3, "importance_pct", importance_pct)
    
    feature_descriptions.append(desc)

feature_description_table = pd.concat(
    feature_descriptions,
    ignore_index=True
)

feature_description_table

,country_segment,feature,importance,importance_pct,IS_FRAUD_USER,count,mean,std,min,25%,50%,75%,max
0,UK,avg_amount_gbp,0.46,45.74,False,"3,207.00","19,092,185.44","633,827,737.97",0.00,"2,416.05","4,888.03","9,472.44","32,305,515,716.97"
1,UK,avg_amount_gbp,0.46,45.74,True,231.00,"211,068.16","2,626,445.23",100.00,"6,674.17","21,299.52","50,370.04","39,950,113.96"
2,UK,share_currency_GBP,0.28,27.68,False,"3,207.00",0.79,0.27,0.00,0.60,0.95,1.00,1.00
3,UK,share_currency_GBP,0.28,27.68,True,231.00,0.98,0.08,0.20,1.00,1.00,1.00,1.00
4,UK,share_type_BANK_TRANSFER,0.10,9.81,False,"3,207.00",0.05,0.12,0.00,0.00,0.00,0.05,1.00
5,UK,share_type_BANK_TRANSFER,0.10,9.81,True,231.00,0.13,0.14,0.00,0.00,0.09,0.22,0.57
6,UK,outlier_share,0.05,4.87,False,"3,207.00",0.13,0.17,0.00,0.01,0.08,0.18,1.00
7,UK,outlier_share,0.05,4.87,True,231.00,0.35,0.25,0.00,0.14,0.33,0.51,1.00
8,UK,share_type_CARD_PAYMENT,0.04,3.64,False,"3,207.00",0.46,0.29,0.00,0.22,0.53,0.70,1.00
9,UK,share_type_CARD_PAYMENT,0.04,3.64,True,231.00,0.27,0.25,0.00,0.05,0.21,0.44,0.88


In [381]:
uk_country_name = "United Kingdom of Great Britain and Northern Ireland (the)"

df["COUNTRY_SEGMENT"] = np.where(
    df["COUNTRY_NAME"] == uk_country_name,
    "UK",
    "NON_UK"
)

country_currency_concentration = (
    df.groupby("COUNTRY_SEGMENT")
    .agg(
        transactions=("USER_ID", "count"),
        unique_users=("USER_ID", "nunique"),

        gbp_transactions=("CURRENCY", lambda x: (x == "GBP").sum()),

        total_amount_gbp=("AMOUNT_GBP", "sum"),
        fraud_amount_gbp=("AMOUNT_GBP", lambda x: x[df.loc[x.index, "IS_FRAUD"]].sum()),

        fraud_transactions=("IS_FRAUD", "sum"),
        fraud_unique_users=("USER_ID", lambda x: df.loc[x.index][df.loc[x.index, "IS_FRAUD"]]["USER_ID"].nunique())
    )
    .reset_index()
)

# Share of total dataset
country_currency_concentration["transaction_share_pct"] = (
    country_currency_concentration["transactions"] / len(df) * 100
).round(2)

country_currency_concentration["user_share_pct"] = (
    country_currency_concentration["unique_users"] / df["USER_ID"].nunique() * 100
).round(2)

country_currency_concentration["amount_gbp_share_pct"] = (
    country_currency_concentration["total_amount_gbp"] / df["AMOUNT_GBP"].sum() * 100
).round(2)

country_currency_concentration["fraud_amount_gbp_share_pct"] = (
    country_currency_concentration["fraud_amount_gbp"] /
    df.loc[df["IS_FRAUD"], "AMOUNT_GBP"].sum() * 100
).round(2)

country_currency_concentration["fraud_transaction_share_pct"] = (
    country_currency_concentration["fraud_transactions"] /
    df["IS_FRAUD"].sum() * 100
).round(2)

country_currency_concentration["fraud_unique_user_share_pct"] = (
    country_currency_concentration["fraud_unique_users"] /
    df.loc[df["IS_FRAUD"], "USER_ID"].nunique() * 100
).round(2)

# GBP concentration within segment
country_currency_concentration["gbp_transaction_share_within_segment_pct"] = (
    country_currency_concentration["gbp_transactions"] /
    country_currency_concentration["transactions"] * 100
).round(2)

# Fraud rates within segment
country_currency_concentration["transaction_fraud_rate_pct"] = (
    country_currency_concentration["fraud_transactions"] /
    country_currency_concentration["transactions"] * 100
).round(2)

country_currency_concentration["user_fraud_rate_pct"] = (
    country_currency_concentration["fraud_unique_users"] /
    country_currency_concentration["unique_users"] * 100
).round(2)

country_currency_concentration["amount_fraud_rate_pct"] = (
    country_currency_concentration["fraud_amount_gbp"] /
    country_currency_concentration["total_amount_gbp"] * 100
).round(2)

country_currency_concentration

,COUNTRY_SEGMENT,transactions,unique_users,gbp_transactions,total_amount_gbp,fraud_amount_gbp,fraud_transactions,fraud_unique_users,transaction_share_pct,user_share_pct,amount_gbp_share_pct,fraud_amount_gbp_share_pct,fraud_transaction_share_pct,fraud_unique_user_share_pct,gbp_transaction_share_within_segment_pct,transaction_fraud_rate_pct,user_fraud_rate_pct,amount_fraud_rate_pct
0,NON_UK,303308,4123,27908,"18,621,379,869,937.62","28,327,363.53",1455,30,44.04,51.40,69.65,1.07,10.00,10.03,9.20,0.48,0.73,0.00
1,UK,385343,3898,311183,"8,116,060,034,849.60","2,621,539,437.58",13088,269,55.96,48.60,30.35,98.93,90.00,89.97,80.75,3.40,6.90,0.03


In [382]:
user_model_df.head()

,USER_ID,IS_FRAUD_USER,AGE,COUNTRY_NAME,total_transactions,avg_amount_gbp,unique_types,unique_currencies,unique_merchant_countries,cross_border_transactions,different_currency_transactions,outlier_transactions,max_same_amount_count_by_user,max_same_amount_count_by_user_type,cross_border_share,different_currency_share,outlier_share,share_type_ATM,share_type_BANK_TRANSFER,share_type_CARD_PAYMENT,share_type_P2P,share_type_TOPUP,share_currency_AED,share_currency_AUD,share_currency_BTC,share_currency_CAD,share_currency_CHF,share_currency_CZK,share_currency_DKK,share_currency_ETH,share_currency_EUR,share_currency_GBP,share_currency_HKD,share_currency_HUF,share_currency_ILS,share_currency_INR,share_currency_JPY,share_currency_LTC,share_currency_MAD,share_currency_NOK,share_currency_NZD,share_currency_PLN,share_currency_QAR,share_currency_RON,share_currency_SEK,share_currency_SGD,share_currency_THB,share_currency_TRY,share_currency_USD,share_currency_XRP,share_currency_ZAR,share_currency_type_CRYPTO,share_currency_type_FIAT,COUNTRY_SEGMENT
0,000e88bb-d302-4fdc-b757-2b1a2c33e7d6,False,37,Denmark,16,"1,876.31",4,2,2,16,3,0,2,2,1.00,0.19,0.00,0.00,0.06,0.50,0.12,0.31,0.00,0.00,0.00,0.00,0.00,0.00,0.81,0.00,0.19,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,NON_UK
1,001032e0-8071-4baf-95b9-e50214665c2e,False,49,Spain,36,"4,788.37",3,2,4,27,36,7,6,6,0.75,1.00,0.19,0.06,0.00,0.64,0.00,0.31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.69,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.31,0.00,0.00,0.00,1.00,NON_UK
2,00131af8-66f0-4526-8b5f-dc2fdb26c7d7,False,40,Portugal,7,"1,618.15",2,1,2,4,7,0,2,2,0.57,1.00,0.00,0.00,0.00,0.43,0.00,0.57,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,NON_UK
3,001926be-3245-43fa-86dd-b40ee160b6f9,False,50,Gibraltar,254,"5,681.80",4,2,4,249,254,33,18,12,0.98,1.00,0.13,0.08,0.00,0.63,0.06,0.23,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.29,0.71,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,NON_UK
4,001cc034-5730-47c6-a70c-25f42249c9ee,False,40,Romania,4,"1,571.02",2,2,1,4,2,0,2,2,1.00,0.50,0.00,0.00,0.50,0.00,0.00,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,NON_UK


#### Bonus Challange

In [383]:
df.head()

,USER_ID,TYPE,CURRENCY,KYC,IS_FRAUD,COUNTRY_NAME,MERCHANT_COUNTRY_NAME,CURRENCY_COUNTRY_NAME,CROSS_BORDER_TRANSACTION,DIFFERENT_CURRENCY_TRANSACTION,AMOUNT_GBP,CURRENCY_TYPE,FLAG_OUTLIER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER,SAME_AMOUNT_GBP_TX_COUNT_BY_USER_TYPE,AGE,COUNTRY_SEGMENT
0,7285c1ec-31d0-4022-b311-0ad9227ef7f4,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Australia,United Kingdom of Great Britain and Northern Ireland (the),True,False,"3,738.00",FIAT,False,1,1,63,UK
1,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,588.00,FIAT,False,1,1,38,UK
2,0fe472c9-cf3e-4e43-90f3-a0cfb6a4f1f0,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Ukraine,United Kingdom of Great Britain and Northern Ireland (the),True,False,"1,264.00",FIAT,False,1,1,49,UK
3,20100a1d-12bc-41ed-a5e1-bc46216e9696,CARD_PAYMENT,GBP,PASSED,False,United Kingdom of Great Britain and Northern Ireland (the),Canada,United Kingdom of Great Britain and Northern Ireland (the),True,False,66.00,FIAT,False,1,1,38,UK
4,821014c5-af06-40ff-91f4-77fe7667809f,CARD_PAYMENT,GBP,FAILED,False,United Kingdom of Great Britain and Northern Ireland (the),New Zealand,United Kingdom of Great Britain and Northern Ireland (the),True,False,968.00,FIAT,False,1,1,34,UK


In [396]:
# ============================================================
# User-level fraudster table with prioritisation score
# Bonus challenge: Top 5 Fraudsters for Head of Risk
# ============================================================

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# ------------------------------------------------------------
# 1. Copy dataset and create helper columns
# ------------------------------------------------------------

df_fraudster = df[df["KYC"] == "PASSED"].copy()

df_fraudster["FRAUD_AMOUNT"] = np.where(
    df_fraudster["IS_FRAUD"] == True,
    df_fraudster["AMOUNT_GBP"],
    0
)

df_fraudster["ATM_TRANSACTION"] = np.where(
    df_fraudster["TYPE"] == "ATM",
    1,
    0
)

df_fraudster["BANK_TRANSFER_TRANSACTION"] = np.where(
    df_fraudster["TYPE"] == "BANK_TRANSFER",
    1,
    0
)

df_fraudster["ATM_AMOUNT"] = np.where(
    df_fraudster["TYPE"] == "ATM",
    df_fraudster["AMOUNT_GBP"],
    0
)

df_fraudster["BANK_TRANSFER_AMOUNT"] = np.where(
    df_fraudster["TYPE"] == "BANK_TRANSFER",
    df_fraudster["AMOUNT_GBP"],
    0
)

# ------------------------------------------------------------
# 2. Country-level totals to use as country denominators
# ------------------------------------------------------------

country_totals = df_fraudster.groupby("COUNTRY_NAME", dropna=False).agg(
    country_transactions=("USER_ID", "count"),
    country_fraud_transactions=("IS_FRAUD", "sum"),
    country_amount=("AMOUNT_GBP", "sum"),
    country_fraud_amount=("FRAUD_AMOUNT", "sum"),
    country_unique_users=("USER_ID", "nunique"),
    country_fraudster_users=(
        "USER_ID",
        lambda x: df_fraudster.loc[x.index][
            df_fraudster.loc[x.index, "IS_FRAUD"]
        ]["USER_ID"].nunique()
    )
).reset_index()

# ------------------------------------------------------------
# 3. Total dataset denominators
# ------------------------------------------------------------

total_transactions = len(df_fraudster)
total_fraud_transactions = df_fraudster["IS_FRAUD"].sum()
total_amount = df_fraudster["AMOUNT_GBP"].sum()
total_fraud_amount = df_fraudster["FRAUD_AMOUNT"].sum()
total_unique_users = df_fraudster["USER_ID"].nunique()
total_fraudster_users = df_fraudster.loc[
    df_fraudster["IS_FRAUD"], "USER_ID"
].nunique()

# ------------------------------------------------------------
# 4. User-level fraudster table
# ------------------------------------------------------------

fraudster_table = df_fraudster.groupby("USER_ID").agg(
    country_name=("COUNTRY_NAME", "first"),
    kyc_status=("KYC", "first"),

    total_transactions=("USER_ID", "count"),
    fraud_transactions=("IS_FRAUD", "sum"),

    atm_transactions=("ATM_TRANSACTION", "sum"),
    bank_transfer_transactions=("BANK_TRANSFER_TRANSACTION", "sum"),
    atm_amount=("ATM_AMOUNT", "sum"),
    bank_transfer_amount=("BANK_TRANSFER_AMOUNT", "sum"),

    total_amount=("AMOUNT_GBP", "sum"),
    fraud_amount=("FRAUD_AMOUNT", "sum"),

    avg_amount=("AMOUNT_GBP", "mean"),
    median_amount=("AMOUNT_GBP", "median"),
    max_amount=("AMOUNT_GBP", "max"),

    unique_types=("TYPE", "nunique"),
    unique_currencies=("CURRENCY", "nunique"),
    unique_merchant_countries=("MERCHANT_COUNTRY_NAME", "nunique")
).reset_index()

# Keep only users with at least one fraud transaction
fraudster_table = fraudster_table[
    fraudster_table["fraud_transactions"] > 0
].copy()

# Since this is user-level, each row represents one fraudster user
fraudster_table["unique_users"] = 1
fraudster_table["fraudster_users"] = 1

# ------------------------------------------------------------
# 5. Merge country denominators into user-level table
# ------------------------------------------------------------

fraudster_table = fraudster_table.merge(
    country_totals,
    left_on="country_name",
    right_on="COUNTRY_NAME",
    how="left"
)

fraudster_table = fraudster_table.drop(columns=["COUNTRY_NAME"])

# ------------------------------------------------------------
# 6. User-level internal rates
# ------------------------------------------------------------

fraudster_table["fraud_rate_pct"] = (
    fraudster_table["fraud_transactions"] /
    fraudster_table["total_transactions"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["atm_transaction_share_pct"] = (
    fraudster_table["atm_transactions"] /
    fraudster_table["total_transactions"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["bank_transfer_transaction_share_pct"] = (
    fraudster_table["bank_transfer_transactions"] /
    fraudster_table["total_transactions"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["atm_amount_share_pct"] = (
    fraudster_table["atm_amount"] /
    fraudster_table["total_amount"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["bank_transfer_amount_share_pct"] = (
    fraudster_table["bank_transfer_amount"] /
    fraudster_table["total_amount"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

# Cash-out = ATM + BANK_TRANSFER
fraudster_table["cashout_amount"] = (
    fraudster_table["atm_amount"] +
    fraudster_table["bank_transfer_amount"]
)

fraudster_table["cashout_amount_share_pct"] = (
    fraudster_table["cashout_amount"] /
    fraudster_table["total_amount"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

# ------------------------------------------------------------
# 7. Share vs total dataset
# ------------------------------------------------------------

fraudster_table["share_total_transactions_pct"] = (
    fraudster_table["total_transactions"] /
    total_transactions * 100
).round(2)

fraudster_table["share_total_fraud_transactions_pct"] = (
    fraudster_table["fraud_transactions"] /
    total_fraud_transactions * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["share_total_amount_pct"] = (
    fraudster_table["total_amount"] /
    total_amount * 100
).round(2)

fraudster_table["share_total_fraud_amount_pct"] = (
    fraudster_table["fraud_amount"] /
    total_fraud_amount * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

# ------------------------------------------------------------
# 8. Share vs user's country
# ------------------------------------------------------------

fraudster_table["share_country_transactions_pct"] = (
    fraudster_table["total_transactions"] /
    fraudster_table["country_transactions"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["share_country_fraud_transactions_pct"] = (
    fraudster_table["fraud_transactions"] /
    fraudster_table["country_fraud_transactions"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["share_country_amount_pct"] = (
    fraudster_table["total_amount"] /
    fraudster_table["country_amount"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

fraudster_table["share_country_fraud_amount_pct"] = (
    fraudster_table["fraud_amount"] /
    fraudster_table["country_fraud_amount"] * 100
).replace([np.inf, -np.inf], 0).fillna(0).round(2)

# ------------------------------------------------------------
# 9. Global ranks
# ------------------------------------------------------------

fraudster_table["rank_global_fraud_amount"] = fraudster_table["fraud_amount"].rank(
    ascending=False,
    method="dense"
).astype(int)

fraudster_table["rank_global_avg_amount"] = fraudster_table["avg_amount"].rank(
    ascending=False,
    method="dense"
).astype(int)

fraudster_table["rank_global_country_fraud_amount_share"] = fraudster_table[
    "share_country_fraud_amount_pct"
].rank(
    ascending=False,
    method="dense"
).astype(int)

fraudster_table["rank_global_cashout_amount_share"] = fraudster_table[
    "cashout_amount_share_pct"
].rank(
    ascending=False,
    method="dense"
).astype(int)

# ------------------------------------------------------------
# 10. Country rank by fraud amount
# ------------------------------------------------------------

fraudster_table["rank_country_fraud_amount"] = (
    fraudster_table
    .groupby("country_name")["fraud_amount"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

fraudster_table["max_country_fraud_amount_rank"] = (
    fraudster_table
    .groupby("country_name")["rank_country_fraud_amount"]
    .transform("max")
    .astype(int)
)

fraudster_table["country_fraud_amount_rank_position"] = (
    fraudster_table["rank_country_fraud_amount"].astype(str) +
    " / " +
    fraudster_table["max_country_fraud_amount_rank"].astype(str)
)

# ------------------------------------------------------------
# 11. Priority score
# No fraud purity logic, because fraud is tagged at user level.
# Higher score = higher priority.
# ------------------------------------------------------------

fraudster_table["score_fraud_amount"] = fraudster_table["fraud_amount"].rank(
    ascending=True,
    pct=True
)

fraudster_table["score_avg_amount"] = fraudster_table["avg_amount"].rank(
    ascending=True,
    pct=True
)

fraudster_table["score_country_fraud_amount_share"] = fraudster_table[
    "share_country_fraud_amount_pct"
].rank(
    ascending=True,
    pct=True
)

fraudster_table["score_cashout_amount"] = fraudster_table[
    "cashout_amount"
].rank(
    ascending=True,
    pct=True
)

fraudster_table["score_cashout_amount_share"] = fraudster_table[
    "cashout_amount_share_pct"
].rank(
    ascending=True,
    pct=True
)

fraudster_table["priority_score"] = (
    fraudster_table["score_fraud_amount"] * 0.20 +
    fraudster_table["score_avg_amount"] * 0.25 +
    fraudster_table["score_country_fraud_amount_share"] * 0.05 +
    fraudster_table["score_cashout_amount"] * 0.35 +
    fraudster_table["score_cashout_amount_share"] * 0.10
).round(4)

fraudster_table["rank_priority"] = fraudster_table["priority_score"].rank(
    ascending=False,
    method="dense"
).astype(int)

# ------------------------------------------------------------
# 12. Final analytical view
# ------------------------------------------------------------

fraudster_table_view = fraudster_table.sort_values(
    [
        "priority_score",
        "fraud_amount",
        "avg_amount",
        "share_country_fraud_amount_pct",
        "cashout_amount_share_pct"
    ],
    ascending=[False, False, False, False, False]
).reset_index(drop=True)

fraudster_table_view = fraudster_table_view[
    [
        "USER_ID",
        "country_name",
        "kyc_status",

        "priority_score",
        "rank_priority",
        "score_fraud_amount",
        "score_avg_amount",
        "score_country_fraud_amount_share",
        "score_cashout_amount",
        "score_cashout_amount_share",

        #"total_transactions",
        "fraud_transactions",
        "fraud_rate_pct",

        #"total_amount",
        "fraud_amount",
        "avg_amount",
        "median_amount",
        "max_amount",

        "share_total_fraud_amount_pct",
        "share_country_fraud_amount_pct",

        "cashout_amount",
        "cashout_amount_share_pct",
        "atm_amount_share_pct",
        "bank_transfer_amount_share_pct",

        "rank_global_fraud_amount",
        "rank_global_avg_amount",
        "rank_global_country_fraud_amount_share",
        "rank_global_cashout_amount_share",
        "country_fraud_amount_rank_position",

        "unique_types",
        "unique_currencies",
        "unique_merchant_countries"
    ]
]

fraudster_table_view.head()

,USER_ID,country_name,kyc_status,priority_score,rank_priority,score_fraud_amount,score_avg_amount,score_country_fraud_amount_share,score_cashout_amount,score_cashout_amount_share,fraud_transactions,fraud_rate_pct,fraud_amount,avg_amount,median_amount,max_amount,share_total_fraud_amount_pct,share_country_fraud_amount_pct,cashout_amount,cashout_amount_share_pct,atm_amount_share_pct,bank_transfer_amount_share_pct,rank_global_fraud_amount,rank_global_avg_amount,rank_global_country_fraud_amount_share,rank_global_cashout_amount_share,country_fraud_amount_rank_position,unique_types,unique_currencies,unique_merchant_countries
0,310667d1-92c6-4fc7-83f1-55b61773ace5,United Kingdom of Great Britain and Northern Ireland (the),PASSED,0.91,1,1.00,0.98,0.89,1.00,0.69,60,100.00,"11,826,891.96","197,114.87","1,995.00","2,072,685.14",0.88,0.89,"5,863,069.96",49.57,0.00,49.57,2,6,25,71,2 / 230,4,3,3
1,88c417b7-c081-4b37-a88a-c5b368ba0a08,United Kingdom of Great Britain and Northern Ireland (the),PASSED,0.89,2,0.96,0.95,0.85,0.97,0.82,37,100.00,"4,627,781.00","125,075.16","80,000.00","2,238,241.00",0.34,0.35,"2,334,200.00",50.44,42.79,7.65,11,15,32,47,11 / 230,4,1,2
2,40a70d7d-447d-426c-872e-012658b2b98c,United Kingdom of Great Britain and Northern Ireland (the),PASSED,0.89,3,0.97,0.89,0.86,0.98,0.88,53,100.00,"4,801,988.00","90,603.55","70,000.00","850,000.00",0.36,0.36,"2,479,000.00",51.62,51.62,0.00,10,29,31,31,10 / 230,3,1,2
3,4f41a015-5205-467d-98d6-a90d343e8afb,United Kingdom of Great Britain and Northern Ireland (the),PASSED,0.89,4,0.96,0.89,0.85,0.98,0.92,51,100.00,"4,598,174.00","90,160.27","50,000.00","2,000,000.00",0.34,0.35,"2,437,000.00",53.00,53.00,0.00,12,30,32,23,12 / 230,3,1,2
4,c1192087-81c9-4328-89a0-398f33cb91de,United Kingdom of Great Britain and Northern Ireland (the),PASSED,0.88,5,0.93,1.00,0.83,0.94,0.72,12,100.00,"4,002,126.00","333,510.50","26,000.00","2,000,000.00",0.30,0.30,"1,991,500.00",49.76,1.10,48.66,18,2,36,63,17 / 230,4,1,2
